# Mission 4: Industrial Complex Segmentation with Multi-Modal Data
## SegFormer-B5 + CMNeXt Fusion + 1x1 Conv Adapter

**Architecture:**
- Main Input: 4-channel (RGB + NIR) satellite imagery
- Auxiliary Input: 48-channel environmental data (CO, NO2, SO2, GEMS)
- Backbone: MiT-B5 with 1x1 Conv Adapter (RGB Identity Initialization)
- Decoder: CMNeXt Fusion Head (multi-scale feature fusion)

**Key Features:**
- Multi-modal data fusion
- Pretrained backbone utilization
- Custom transforms for label remapping
- Wandb logging for experiment tracking

## 1. Environment Setup

In [1]:
cd ../

/home/work/root/Meme_coin/ddc


In [2]:
cd mmsegmentation

/home/work/root/Meme_coin/ddc/mmsegmentation


In [1]:
import sys
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation")
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation/mmseg")

from mmseg.registry import TRANSFORMS, MODELS
from mmcv.transforms import BaseTransform
print("Imported OK")


Imported OK


## 2. Imports and Configuration

In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import sys
import random
import warnings
import logging
from typing import List, Optional

warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tifffile
# import wandb

from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.model import BaseModule
from mmengine.logging import print_log
from mmseg.registry import TRANSFORMS, MODELS
from mmseg.models.decode_heads.decode_head import BaseDecodeHead
from mmseg.models.segmentors import EncoderDecoder
from mmseg.structures import SegDataSample
from mmcv.transforms import BaseTransform


# Path setup for mmsegmentation
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation")
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation/mmseg")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"Wandb version: {wandb.__version__}")

PyTorch: 2.0.0a0+1767026
CUDA available: True


In [3]:
# ==========================================
# Global Configuration
# ==========================================

# Random Seed
SEED = 42

# Paths
DATA_ROOT = '/home/work/root/Meme_coin/ddc/raw_data/industrial_complex'
EXTRA_DATA_ROOT = '/home/work/root/Meme_coin/ddc/raw_data/extra_data'
WORK_DIR = '/home/work/root/Meme_coin/ddc/work_dirs/segformer_b5_cmnext_m4_final'


# Wandb Configuration
WANDB_PROJECT = 'datacreatorcamp'
WANDB_NAME = 'segformer-b5-cmnext'
WANDB_ENTITY = None  # Set your wandb username/team here if needed

# Training Configuration
MAX_ITERS = 200000
VAL_INTERVAL = 1000
LOG_INTERVAL = 100
CHECKPOINT_INTERVAL = 5000

# Main Image Statistics (4-channel: RGBN)
IMG_MEAN = [2110.71, 2048.15, 1828.06, 3291.53]
IMG_STD = [633.93, 523.48, 533.74, 859.42]


# Environmental Maps Statistics (CO(12) + NO2(12) + SO2(12) + GEMS(12))
CO_MEAN_12 = [    0.069872, 0.071186, 0.069363, 0.0711, 0.072274, 0.079479, 
    0.083693, 0.098444, 0.100171, 0.089323, 0.080561, 0.079747]
NO2_MEAN_12 = [    0.006356, 0.005455, 0.00446, 0.004638, 0.006101, 0.009362, 
    0.009916, 0.011757, 0.01142, 0.006965, 0.008241, 0.007259]
SO2_MEAN_12 = [    0.001406, 0.001333, 0.001179, 0.001219, 0.001285, 0.00147, 
    0.001493, 0.001601, 0.001576, 0.001467, 0.001438, 0.001458]
GEMS_MEAN_12 = [1.0001566707337604e+16, 8884934605021998.0, 6141337346240758.0, 6333264634799439.0,
                8567805078029439.0, 1.5086685790860244e+16, 1.895145309213309e+16, 2.4798531125954364e+16,
                2.3445578420653668e+16, 1.1999405246184784e+16, 1.5631455660836568e+16, 1.147605622779386e+16]

CO_STD_12 = [    0.135525, 0.137976, 0.134637, 0.137605, 0.139468, 0.153574, 
    0.162024, 0.189629, 0.193217, 0.172882, 0.155895, 0.154963]
NO2_STD_12 = [    0.00291, 0.002429, 0.002454, 0.00209, 0.001952, 0.002244, 
    0.002634, 0.003307, 0.003251, 0.003801, 0.0025, 0.00314]
SO2_STD_12 = [    0.000531, 0.000528, 0.000596, 0.000588, 0.000503, 0.000475, 
    0.000526, 0.00051, 0.000591, 0.00051, 0.000553, 0.000618]
GEMS_STD_12 = [2652378441960289.5, 2308996059307467.5, 1644847904756469.8, 1755220855234594.2,
               2413921046387099.0, 4865034759856011.0, 5792696868603161.0, 9519573561097318.0,
               8308107813709501.0, 4209704129784980.0, 4740854322705774.0, 3306107477728583.5]



os.makedirs(WORK_DIR, exist_ok=True)
print(f"✅ Configuration loaded")
print(f"   - Work directory: {WORK_DIR}")
print(f"   - Wandb project: {WANDB_PROJECT}")
print(f"   - Experiment name: {WANDB_NAME}")

✅ Configuration loaded
   - Work directory: /home/work/root/Meme_coin/ddc/work_dirs/segformer_b5_cmnext_m4_final
   - Wandb project: datacreatorcamp
   - Experiment name: segformer-b5-cmnext


In [4]:
# ==========================================
# Seed Fixing for Reproducibility
# ==========================================

def set_seed(seed=42):
    """
    Fix all random seeds for reproducible results
    
    Args:
        seed (int): Random seed value
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    print(f"✅ Random seed fixed to {seed}")

set_seed(SEED)

✅ Random seed fixed to 42


## 3. Custom Transforms and Data Loaders

In [5]:
# ==========================================
# Custom Transforms
# ==========================================
from mmengine.registry import TRANSFORMS
from mmseg.structures import SegDataSample

@TRANSFORMS.register_module()
class RemapLabels(BaseTransform):
    """
    Remap label values: Industrial Complex(10→1), Background(90→0)
    """
    def __init__(self, original_values=(10, 90), target_values=(1, 0)):
        self.original_values = original_values
        self.target_values = target_values
    
    def transform(self, results):
        if 'gt_seg_map' in results:
            seg_map = results['gt_seg_map']
            new_seg_map = seg_map.copy()
            
            for orig_val, target_val in zip(self.original_values, self.target_values):
                new_seg_map[seg_map == orig_val] = target_val
            
            results['gt_seg_map'] = new_seg_map
        
        return results



@TRANSFORMS.register_module()
class LoadMultiModalData(BaseTransform):
    """
    Load main image + environmental maps (CO, NO2, SO2, GEMS)
    
    주의: GEMS는 1e16 스케일이므로 float64로 로드해야 정보 손실 없음
         정규화 후에는 float32로 변환 가능 (정규화 시 -3~+3 범위로 스케일 조정됨)
    """
    def __init__(self, extra_data_root: str, imdecode_backend: str = 'tifffile'):
        self.extra_data_root = extra_data_root
        if imdecode_backend != 'tifffile':
            raise ValueError("Only 'tifffile' backend is supported")

    def _load_image(self, filepath: str, use_float64: bool = False) -> np.ndarray:
        """
        Args:
            filepath: 이미지 파일 경로
            use_float64: True일 경우 float64로 로드 (GEMS용, 1e16 스케일 보존)
        """
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")
        
        img = tifffile.imread(filepath)
        
        # 채널 축 정리: (H,W,C) → (C,H,W)
        if img.ndim == 3 and img.shape[2] > 1:
            img = np.transpose(img, (2, 0, 1))
        elif img.ndim == 2:
            img = img[None, ...]
        
        # 타입 변환
        dtype = np.float64 if use_float64 else np.float32
        return img.astype(dtype, copy=False)

    def _infer_split_and_prefix(self, main_img_path: str):
        fname = os.path.basename(main_img_path)
        if fname.startswith('TS_'):
            return 'train', 'TS_'
        if fname.startswith('VS_'):
            return 'val', 'VS_'
        if f'{os.sep}train{os.sep}' in main_img_path:
            return 'train', 'TS_'
        if f'{os.sep}val{os.sep}' in main_img_path:
            return 'val', 'VS_'
        return 'train', 'TS_'

    def transform(self, results: dict) -> dict:
        # 1) 메인 이미지
        main_img_path = results['img_path']
        main_img = self._load_image(main_img_path)  # float32
        results['img'] = main_img
        results['img_shape'] = main_img.shape[1:]
        results['ori_shape'] = main_img.shape[1:]

        # 2) 경로 구성
        split, prefix = self._infer_split_and_prefix(main_img_path)
        filename = os.path.basename(main_img_path)
        suffix = filename[len(prefix):] if filename.startswith(prefix) else filename

        base_env_path = os.path.join(self.extra_data_root, split, 'img')
        air_dir  = f'{prefix}SN10_AIR_Pollution'
        gems_dir = f'{prefix}SN10_GEMS'

        # 3) 환경 데이터 로드
        # CO, NO2, SO2는 작은 스케일(0.001~0.1)이므로 float32 충분
        co   = self._load_image(
            os.path.join(base_env_path, air_dir, f'AIR_Pollution_CO_{suffix}')
        )
        no2  = self._load_image(
            os.path.join(base_env_path, air_dir, f'AIR_Pollution_NO2_{suffix}')
        )
        so2  = self._load_image(
            os.path.join(base_env_path, air_dir, f'AIR_Pollution_SO2_{suffix}')
        )
        
        # ✅ GEMS만 float64로 로드 (1e16 스케일 → float32는 정밀도 부족)
        gems = self._load_image(
            os.path.join(base_env_path, gems_dir, f'GEMS_{suffix}'),
            use_float64=True
        )

        # 4) 결과 저장
        results['env_co']   = co    # float32
        results['env_no2']  = no2   # float32
        results['env_so2']  = so2   # float32
        results['env_gems'] = gems  # float64 (정규화 전까지 유지)
        
        return results


@TRANSFORMS.register_module()
class NormalizeByKey(BaseTransform):
    """
    Normalize multiple keys with different statistics
    
    처리 흐름:
    1. 입력: env_gems는 float64, 나머지는 float32
    2. 계산: 모두 float64로 변환하여 정규화 (1e16 스케일 안정 처리)
    3. 출력: float32로 반환 (정규화 후 -3~+3 범위이므로 정밀도 충분)
    """
    def __init__(self, specs: dict, eps: float = 1e-6):
        self.eps = float(eps)
        self.specs = {}
        for k, (m, s) in specs.items():
            # 통계는 64비트로 보관
            m = np.asarray(m, dtype=np.float64)
            s = np.asarray(s, dtype=np.float64)
            assert m.ndim == 1 and s.ndim == 1 and m.shape == s.shape, \
                   f"{k} mean/std shape mismatch"
            self.specs[k] = (m, s)

    def transform(self, results: dict) -> dict:
        for key, (mean, std) in self.specs.items():
            if key not in results:
                continue

            x = results[key]
            assert x.ndim == 3 and x.shape[0] == mean.shape[0], \
                   f"{key} shape {x.shape}"

            # NaN/Inf 처리 + float64 변환
            # env_gems는 이미 float64, 나머지는 float32→float64 변환
            x64 = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
            
            m64 = mean[:, None, None]
            s64 = np.maximum(std, self.eps)[:, None, None]
            
            # 정규화: (x - mean) / std
            # GEMS: (1e16 - 1e16) / 2.6e15 → -3~+3 범위로 스케일 조정
            x64 = (x64 - m64) / s64
            
            # ✅ 정규화 후 float32 변환 (정보 손실 거의 없음)
            # 이유: 정규화된 값은 대부분 -3~+3 범위, float32 정밀도 충분
            results[key] = x64.astype(np.float32, copy=False)
        
        return results




@TRANSFORMS.register_module()
class PackSegInputsKeepEnv(BaseTransform):
    def __init__(
        self,
        meta_keys=('img_path', 'ori_shape', 'img_shape'),
        env_keys=('env_co', 'env_no2', 'env_so2', 'env_gems'),
        out_key=None,
    ):
        self.meta_keys = tuple(meta_keys)
        self.env_keys  = tuple(env_keys)
        self.out_key   = out_key

    def transform(self, results):
        from mmengine.structures import PixelData  # ✅ 추가
        
        sample = SegDataSample()

        # 1) 이미지 / GT
        inputs = torch.from_numpy(results['img'])  # (C,H,W)
        
        # ✅ 수정된 부분
        if 'gt_seg_map' in results:
            gt_sem_seg_data = dict(
                data=torch.from_numpy(results['gt_seg_map']).long()
            )
            sample.gt_sem_seg = PixelData(**gt_sem_seg_data)

        # 2) 메타정보
        metainfo = {k: results[k] for k in self.meta_keys if k in results}
        sample.set_metainfo(metainfo)

        # 3) 환경 모달 분리 입력 채우기 (헤드가 바로 읽음)
        env_list = []
        for k in self.env_keys:
            assert k in results, f"Missing env key in results: {k}"
            t = torch.from_numpy(results[k])
            if t.ndim == 2:
                t = t.unsqueeze(0)
            setattr(sample, k, t)
            env_list.append(t)

        sample.env_streams = env_list

        if self.out_key is not None:
            sample.__setattr__(self.out_key, torch.cat(env_list, dim=0))

        # 4) 반환
        return dict(inputs=inputs, data_samples=sample)


print("✅ Custom transforms registered")

✅ Custom transforms registered


## 4. Model Architecture

In [ ]:
# ==========================================
# CMNeXt-Style Fusion on MiT-B5 Backbone
#  - FRM / FFM: CMNeXt 원본 구조 그대로 사용
#  - Backbone: MiT-B5 (mmseg MixVisionTransformer)
# ==========================================
import math
from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.init import trunc_normal_

from mmengine.model import BaseModule
from mmseg.registry import MODELS
from mmseg.structures import SegDataSample
from mmseg.models.segmentors.encoder_decoder import EncoderDecoder


# ==========================================
# 0. Utils
# ==========================================
def _resize_like(x: torch.Tensor, ref: torch.Tensor, mode='bilinear', align_corners=False):
    if x.shape[2:] != ref.shape[2:]:
        return F.interpolate(x, size=ref.shape[2:], mode=mode, align_corners=align_corners)
    return x


# ---------- SegFormer-style LinearEmbed (필요시) ----------
class LinearEmbed(nn.Module):
    """(B,C,H,W) -> (B,H*W,embed_dim)"""
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=1, bias=True)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x


# ==========================================
# 1. CMNeXt 원본 FRM / FFM
#    (FeatureRectifyModule, FeatureFusionModule)
# ==========================================

# ----- ChannelWeights / SpatialWeights -----
class ChannelWeights(nn.Module):
    def __init__(self, dim, reduction=1):
        super(ChannelWeights, self).__init__()
        self.dim = dim
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(self.dim * 4, self.dim * 4 // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(self.dim * 4 // reduction, self.dim * 2),
            nn.Sigmoid()
        )

    def forward(self, x1, x2):
        B, _, H, W = x1.shape
        x = torch.cat((x1, x2), dim=1)          # (B, 2C, H, W)
        avg = self.avg_pool(x).view(B, self.dim * 2)
        maxv = self.max_pool(x).view(B, self.dim * 2)
        y = torch.cat((avg, maxv), dim=1)       # (B, 4C)
        y = self.mlp(y).view(B, self.dim * 2, 1)
        # → (2, B, C, 1, 1)
        channel_weights = y.reshape(B, 2, self.dim, 1, 1).permute(1, 0, 2, 3, 4)
        return channel_weights


class SpatialWeights(nn.Module):
    def __init__(self, dim, reduction=1):
        super(SpatialWeights, self).__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Conv2d(self.dim * 2, self.dim // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(self.dim // reduction, 2, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x1, x2):
        B, _, H, W = x1.shape
        x = torch.cat((x1, x2), dim=1)          # (B, 2C, H, W)
        spatial_weights = self.mlp(x).reshape(B, 2, 1, H, W).permute(1, 0, 2, 3, 4)
        return spatial_weights


class FeatureRectifyModule(nn.Module):
    """
    CMNeXt / CMX의 CM-FRM 그대로 + 기여도 통계(optional):
      out_x1 = x1 + λc * Wc(x2) + λs * Ws(x2)
      out_x2 = x2 + λc * Wc(x1) + λs * Ws(x1)

    통계(가정: x1=RGB, x2=Env 계열):
      - env_to_rgb_mag: Env→RGB 보정항의 평균 |값|
      - rgb_to_env_mag: RGB→Env 보정항의 평균 |값|
    """
    def __init__(self, dim, reduction=1, lambda_c=.5, lambda_s=.5,
                 track_stats: bool = False):
        super(FeatureRectifyModule, self).__init__()
        self.lambda_c = lambda_c
        self.lambda_s = lambda_s
        self.channel_weights = ChannelWeights(dim=dim, reduction=reduction)
        self.spatial_weights = SpatialWeights(dim=dim, reduction=reduction)
        self.track_stats = track_stats

        # 통계용 버퍼
        self.register_buffer('env_to_rgb_mag', torch.zeros(1), persistent=False)
        self.register_buffer('rgb_to_env_mag', torch.zeros(1), persistent=False)
        self.register_buffer('frm_calls', torch.zeros(1), persistent=False)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x1, x2):
        """
        x1, x2: (B, C, H, W)
        여기서는 x1=RGB branch, x2=Env branch라고 가정
        """
        channel_weights = self.channel_weights(x1, x2)   # (2, B, C, 1, 1)
        spatial_weights = self.spatial_weights(x1, x2)   # (2, B, 1, H, W)

        # 보정항 분리
        contrib_x1 = self.lambda_c * channel_weights[1] * x2 + \
                     self.lambda_s * spatial_weights[1] * x2  # Env→RGB
        contrib_x2 = self.lambda_c * channel_weights[0] * x1 + \
                     self.lambda_s * spatial_weights[0] * x1  # RGB→Env

        out_x1 = x1 + contrib_x1
        out_x2 = x2 + contrib_x2

        # ---------- 통계 업데이트 ----------
        if self.track_stats:
            with torch.no_grad():
                # 평균 |값|으로 magnitude 측정
                env_to_rgb = contrib_x1.abs().mean()
                rgb_to_env = contrib_x2.abs().mean()
                self.env_to_rgb_mag += env_to_rgb
                self.rgb_to_env_mag += rgb_to_env
                self.frm_calls += 1

        return out_x1, out_x2



# ----- Stage1: CrossPath (efficient cross-attention) -----
class CrossAttention(nn.Module):
    """
    CMNeXt Stage1에서 쓰는 efficient cross-attention
      - q1, q2: 두 모달의 query
      - k1,v1 / k2,v2: 각 모달 자체의 kv로 global context 생성
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None):
        super(CrossAttention, self).__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.kv1 = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.kv2 = nn.Linear(dim, dim * 2, bias=qkv_bias)

    def forward(self, x1, x2):
        B, N, C = x1.shape
        q1 = x1.reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3).contiguous()
        q2 = x2.reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3).contiguous()
        k1, v1 = self.kv1(x1).reshape(B, N, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4).contiguous()
        k2, v2 = self.kv2(x2).reshape(B, N, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4).contiguous()

        ctx1 = (k1.transpose(-2, -1) @ v1) * self.scale
        ctx1 = ctx1.softmax(dim=-2)
        ctx2 = (k2.transpose(-2, -1) @ v2) * self.scale
        ctx2 = ctx2.softmax(dim=-2)

        x1 = (q1 @ ctx2).permute(0, 2, 1, 3).reshape(B, N, C).contiguous()
        x2 = (q2 @ ctx1).permute(0, 2, 1, 3).reshape(B, N, C).contiguous()
        return x1, x2


class CrossPath(nn.Module):
    """
    CMNeXt 식 (7)(8)에 대응:
      - 채널 축에서 residual/inter 활성 분리
      - 활성 부분(u1,u2)만 cross-attention
    """
    def __init__(self, dim, reduction=1, num_heads=None, norm_layer=nn.LayerNorm):
        super().__init__()
        self.channel_proj1 = nn.Linear(dim, dim // reduction * 2)
        self.channel_proj2 = nn.Linear(dim, dim // reduction * 2)
        self.act1 = nn.ReLU(inplace=True)
        self.act2 = nn.ReLU(inplace=True)
        self.cross_attn = CrossAttention(dim // reduction, num_heads=num_heads)
        self.end_proj1 = nn.Linear(dim // reduction * 2, dim)
        self.end_proj2 = nn.Linear(dim // reduction * 2, dim)
        self.norm1 = norm_layer(dim)
        self.norm2 = norm_layer(dim)

    def forward(self, x1, x2):
        # x1,x2: (B,N,C)
        y1, u1 = self.act1(self.channel_proj1(x1)).chunk(2, dim=-1)
        y2, u2 = self.act2(self.channel_proj2(x2)).chunk(2, dim=-1)
        v1, v2 = self.cross_attn(u1, u2)
        y1 = torch.cat((y1, v1), dim=-1)
        y2 = torch.cat((y2, v2), dim=-1)
        out_x1 = self.norm1(x1 + self.end_proj1(y1))
        out_x2 = self.norm2(x2 + self.end_proj2(y2))
        return out_x1, out_x2


# ----- Stage2: ChannelEmbed + FFM -----
class ChannelEmbed(nn.Module):
    def __init__(self, in_channels, out_channels, reduction=1, norm_layer=nn.BatchNorm2d):
        super(ChannelEmbed, self).__init__()
        self.out_channels = out_channels
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.channel_embed = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // reduction, kernel_size=1, bias=True),
            nn.Conv2d(out_channels // reduction, out_channels // reduction,
                      kernel_size=3, stride=1, padding=1, bias=True,
                      groups=out_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // reduction, out_channels, kernel_size=1, bias=True),
            norm_layer(out_channels)
        )
        self.norm = norm_layer(out_channels)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.permute(0, 2, 1).reshape(B, C, H, W).contiguous()
        residual = self.residual(x)
        x = self.channel_embed(x)
        out = self.norm(residual + x)
        return out


class FeatureFusionModule(nn.Module):
    """
    CMNeXt / CMX의 CM-FFM 그대로:
      - Stage1: CrossPath (efficient cross-attention)
      - Stage2: ChannelEmbed로 두 모달 fused map 생성
    """
    def __init__(self, dim, reduction=1, num_heads=None, norm_layer=nn.BatchNorm2d):
        super().__init__()
        self.cross = CrossPath(dim=dim, reduction=reduction, num_heads=num_heads)
        self.channel_emb = ChannelEmbed(
            in_channels=dim * 2,
            out_channels=dim,
            reduction=reduction,
            norm_layer=norm_layer
        )
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x1, x2):
        """
        x1, x2: (B,C,H,W)
        """
        B, C, H, W = x1.shape
        x1 = x1.flatten(2).transpose(1, 2)    # (B,N,C)
        x2 = x2.flatten(2).transpose(1, 2)
        x1, x2 = self.cross(x1, x2)
        merge = torch.cat((x1, x2), dim=-1)   # (B,N,2C)
        merge = self.channel_emb(merge, H, W) # (B,C,H,W)
        return merge


# ==========================================
# 2. CMNeXt: Self-Query Hub + PPX + Aux Stem
# ==========================================
class SelfQueryHub(nn.Module):
    """
    CMNeXt-style 모달 허브 + 선택 통계(optional):
      - 각 모달에 DWConv3x3 + score conv(1x1, sigmoid)
      - f_m + q_m * DWConv(f_m)
      - 위치별 Q_m 최대 모달 선택 → f^q

    통계:
      - track_stats=True 일 때
        self.select_counts: (M,)  각 모달이 선택된 총 횟수 (pixel 단위)
        self.total_positions: 스칼라, 전체 선택 수 (B*H*W 누적)
    """
    def __init__(self, c, track_stats: bool = False):
        super().__init__()
        self.dw = nn.Conv2d(c, c, 3, padding=1, groups=c, bias=True)
        self.sc = nn.Conv2d(c, 1, 1, bias=True)

        self.track_stats = track_stats
        self.num_modalities: Optional[int] = None
        # 통계용 버퍼 (lazy init)
        self.register_buffer('select_counts', torch.zeros(1), persistent=False)
        self.register_buffer('total_positions', torch.zeros(1), persistent=False)

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        cands, scores = [], []
        for fm in feats:
            fhat = self.dw(fm)
            q = torch.sigmoid(self.sc(fhat))        # (B,1,H,W)
            cands.append(fm + q * fhat)             # (B,C,H,W)
            scores.append(q.squeeze(1))             # (B,H,W)

        score_map = torch.stack(scores, dim=1)      # (B,M,H,W)
        idx = score_map.argmax(dim=1, keepdim=True) # (B,1,H,W)
        stack = torch.stack(cands, dim=1)           # (B,M,C,H,W)
        B, M, C, H, W = stack.shape

        # ---------- 통계 업데이트 ----------
        if self.track_stats:
            with torch.no_grad():
                if (self.num_modalities is None) or (self.num_modalities != M):
                    self.num_modalities = M
                    self.select_counts = torch.zeros(
                        M, device=stack.device, dtype=torch.float32
                    )
                    self.total_positions = torch.zeros(
                        1, device=stack.device, dtype=torch.float32
                    )
                chosen = idx.squeeze(1).reshape(-1).to(torch.long)  # (B*H*W,)
                binc = torch.bincount(chosen, minlength=M).to(self.select_counts.device)
                self.select_counts += binc.float()
                self.total_positions += float(chosen.numel())

        # ---------- 실제 선택 ----------
        idx_expanded = idx.unsqueeze(2).expand(B, 1, C, H, W)
        fq = torch.gather(stack, 1, idx_expanded).squeeze(1)  # (B,C,H,W)
        return fq



class PPX(nn.Module):
    """
    CMNeXt Parallel Pooling Mixer + SE:
      - DWConv7x7
      - k∈{3,7,11} avg pool sum
      - 1×1 conv + sigmoid → w
      - (1 + w) * f^q 후 SE
    """
    def __init__(self, c, pool_ks=(3, 7, 11)):
        super().__init__()
        self.dw7 = nn.Conv2d(c, c, 7, padding=3, groups=c, bias=True)
        self.mix = nn.Conv2d(c, c, 1, bias=True)
        self.se  = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, max(1, c // 4), 1), nn.ReLU(inplace=True),
            nn.Conv2d(max(1, c // 4), c, 1), nn.Sigmoid()
        )
        self.pool_ks = pool_ks

    def forward(self, fq: torch.Tensor) -> torch.Tensor:
        x = self.dw7(fq)
        s = x
        for k in self.pool_ks:
            s = s + F.avg_pool2d(x, kernel_size=k, stride=1, padding=k // 2)
        w = torch.sigmoid(self.mix(s))
        fw = w * fq + fq
        return fw * self.se(fw)


class AuxModalStem(nn.Module):
    """
    입력: (B,12,64,64)  (예: 12개월 × 1채널 4모달 중 하나)
    출력:
      S2: (B,c2,64,64)
      S3: (B,c3,32,32)
      S4: (B,c4,16,16)
    """
    def __init__(self, c2=128, c3=320, c4=512, in_ch=12):
        super().__init__()
        self.s2 = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, c2, 3, padding=1, bias=False),
            nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
        )
        self.s3 = nn.Sequential(
            nn.Conv2d(c2, c3, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
        )
        self.s4 = nn.Sequential(
            nn.Conv2d(c3, c4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
        )

    def forward(self, x12):
        f2 = self.s2(x12)     # 64×64
        f3 = self.s3(f2)      # 32×32
        f4 = self.s4(f3)      # 16×16
        return f2, f3, f4


# ==========================================
# 3. Backbone: MiT-B5 + Hub2Fuse-in-Encoder
#    (FRM/FFM = CMNeXt 원본)
# ==========================================
@MODELS.register_module()
class CMNeXtCascadingBackbone(BaseModule):
    """
    주분기: MiT-B5 (RGBN → 1x1Conv → RGB)
    보조분기: Env(48ch) → 4×12ch 모달 → AuxModalStem
    S2/S3/S4에서:
      - SelfQueryHub (modality hub)
      - PPX (multi-scale pooling)
      - FRM (CM-FRM: 채널/공간 보정)
      - FFM (CM-FFM: CrossPath + ChannelEmbed)
      - 복원맵(add)로 다음 stage patch_embed 입력
    """
    def __init__(self,
                 mit_b5_cfg: dict,
                 in_channels: int = 4,        # RGBN
                 aux_c2: Optional[int] = None,
                 aux_c3: Optional[int] = None,
                 aux_c4: Optional[int] = None,
                 init_cfg=None,
                 track_frm_stats: bool = False):
        super().__init__(init_cfg)
        # ----- MiT-B5 본체 -----
        self.mit = MODELS.build(mit_b5_cfg)

        # 4->3 어댑터 (RGB identity, NIR zero)
        assert in_channels in (3, 4)
        self.use_adapter = (in_channels == 4)
        if self.use_adapter:
            self.rgbn_adapter = nn.Conv2d(4, 3, 1, bias=False)
            with torch.no_grad():
                w = self.rgbn_adapter.weight
                w.zero_()
                w[0, 0, 0, 0] = 1.0  # R->R
                w[1, 1, 0, 0] = 1.0  # G->G
                w[2, 2, 0, 0] = 1.0  # B->B

        # MiT-B5 stage 채널
        c1, c2, c3, c4 = 64, 128, 320, 512

        # Aux 채널을 MiT stage 채널과 맞춤
        if aux_c2 is None: aux_c2 = c2
        if aux_c3 is None: aux_c3 = c3
        if aux_c4 is None: aux_c4 = c4

        # 4개의 Aux 모달 stem (CO/NO2/SO2/GEMS 등)
        self.stems = nn.ModuleList([
            AuxModalStem(c2=aux_c2, c3=aux_c3, c4=aux_c4, in_ch=12)
            for _ in range(4)
        ])

        # Stage별 SelfQueryHub / PPX / FRM / FFM
        # num_heads는 CMNeXt 설정과 동일하게 [1,2,5,8] 사용
        self.sq2, self.ppx2, self.frm2, self.ffm2 = (
            SelfQueryHub(c2,track_stats=True),
            PPX(c2),
            FeatureRectifyModule(dim=c2, reduction=1, track_stats=track_frm_stats),
            FeatureFusionModule(dim=c2, reduction=1, num_heads=2, norm_layer=nn.BatchNorm2d)
        )
        self.sq3, self.ppx3, self.frm3, self.ffm3 = (
            SelfQueryHub(c3,track_stats=True),
            PPX(c3),
            FeatureRectifyModule(dim=c3, reduction=1 ,track_stats=track_frm_stats),
            FeatureFusionModule(dim=c3, reduction=1, num_heads=5, norm_layer=nn.BatchNorm2d)
        )
        self.sq4, self.ppx4, self.frm4, self.ffm4 = (
            SelfQueryHub(c4,track_stats=True),
            PPX(c4),
            FeatureRectifyModule(dim=c4, reduction=1, track_stats=track_frm_stats),
            FeatureFusionModule(dim=c4, reduction=1, num_heads=8, norm_layer=nn.BatchNorm2d)
        )

        # MiT 내부 핸들 바인딩
        self._bind_mit_handles()

    # ---------- MiT 핸들 바인딩 ----------
    def _bind_mit_handles(self):
        m = self.mit
        assert hasattr(m, 'layers'), \
            "MixVisionTransformer(1.1.1)는 self.layers[i] = [patch_embed, blocks, norm] 구조여야 합니다."

        # MiT 내부 모듈에 대한 '핸들'만 리스트에 저장 (submodule 등록 X)
        self.pe, self.bl, self.nm = [], [], []
        for i in range(4):
            patch_embed_i, blocks_i, norm_i = m.layers[i]
            self.pe.append(patch_embed_i)
            self.bl.append(blocks_i)
            self.nm.append(norm_i)



    # ---------- 유틸 ----------
    @staticmethod
    def _tokens_to_map(x, hw):  # (B,HW,C) -> (B,C,H,W)
        H, W = hw
        B, _, C = x.shape
        return x.reshape(B, H, W, C).permute(0, 3, 1, 2).contiguous()

    @staticmethod
    def _forward_stage(blocks, x, hw):
        for blk in blocks:
            x = blk(x, hw)   # (x, hw_shape)
        return x

    def _aux_lists(self, env: torch.Tensor):
        """env: (B,48,64,64) -> 각 stage별 AUX 후보 리스트"""
        mods = list(env.split(12, dim=1))  # 4개 모달
        a2_list, a3_list, a4_list = [], [], []
        for x12, stem in zip(mods, self.stems):
            a2, a3, a4 = stem(x12)          # 64,32,16
            a2_list.append(a2)
            a3_list.append(a3)
            a4_list.append(a4)
        return a2_list, a3_list, a4_list

    # ---------- forward ----------
    def forward(self, img: torch.Tensor, env: torch.Tensor):
        """
        img : (B,4,H,W)  - RGBN
        env : (B,48,64,64) - 4모달×12개월 환경 데이터
        """
        # ----- 입력 정리 -----
        if self.use_adapter:
            img = self.rgbn_adapter(img)    # 4->3

        # ==========================
        # 1) 순수 MiT 경로 먼저 싹 돌림 (Env 개입 X)
        # ==========================

        # ----- Stage1 (MiT 원형) -----
        pe1, bl1, nm1 = self.pe[0], self.bl[0], self.nm[0]
        x, hw = pe1(img)                           # (x, (H/4, W/4))
        x = self._forward_stage(bl1, x, hw)
        x = nm1(x)
        s1 = self._tokens_to_map(x, hw)           # (B,64,H/4,W/4)

        # ----- Stage2 (MiT 원형) -----
        pe2, bl2, nm2 = self.pe[1], self.bl[1], self.nm[1]
        x, hw = pe2(s1)
        x = self._forward_stage(bl2, x, hw)
        x = nm2(x)
        s2 = self._tokens_to_map(x, hw)           # (B,128,H/8,W/8)

        # ----- Stage3 (MiT 원형) -----
        pe3, bl3, nm3 = self.pe[2], self.bl[2], self.nm[2]
        x, hw = pe3(s2)
        x = self._forward_stage(bl3, x, hw)
        x = nm3(x)
        s3 = self._tokens_to_map(x, hw)           # (B,320,H/16,W/16)

        # ----- Stage4 (MiT 원형) -----
        pe4, bl4, nm4 = self.pe[3], self.bl[3], self.nm[3]
        x, hw = pe4(s3)
        x = self._forward_stage(bl4, x, hw)
        x = nm4(x)
        s4 = self._tokens_to_map(x, hw)           # (B,512,H/32,W/32)

        # 여기까지가 순수 MiT backbone
        # img -> s1 -> s2 -> s3 -> s4 (Env 완전 분리)

        # ==========================
        # 2) Env branch + 각 stage별 fusion (MiT로 피드백 X)
        # ==========================

        # ----- Aux 후보 생성 -----
        a2_list, a3_list, a4_list = self._aux_lists(env)

        # ----- S2: SQ-Hub → PPX → FRM/FFM → 디코더용 feature -----
        a2_list = [_resize_like(a2, s2) for a2 in a2_list]
        fq2 = self.sq2(a2_list)                   # (B,128,H/8,W/8)
        fw2 = self.ppx2(fq2)
        m2r, q2r = self.frm2(s2, fw2)
        f2 = self.ffm2(m2r, q2r)
        s2_fused = s2 + f2                        # ★ 디코더용

        # ----- S3: SQ-Hub → PPX → FRM/FFM -----
        a3_list = [_resize_like(a3, s3) for a3 in a3_list]
        fq3 = self.sq3(a3_list)                   # (B,320,H/16,W/16)
        fw3 = self.ppx3(fq3)
        m3r, q3r = self.frm3(s3, fw3)
        f3 = self.ffm3(m3r, q3r)
        s3_fused = s3 + f3                        # ★ 디코더용

        # ----- S4: SQ-Hub → PPX → FRM/FFM -----
        a4_list = [_resize_like(a4, s4) for a4 in a4_list]
        fq4 = self.sq4(a4_list)                   # (B,512,H/32,W/32)
        fw4 = self.ppx4(fq4)
        m4r, q4r = self.frm4(s4, fw4)
        f4 = self.ffm4(m4r, q4r)
        s4_fused = s4 + f4                        # ★ 디코더용

        # 디코더 입력 (SegFormerHead 기대 채널: [64,128,320,512])
        #  - s1       : 순수 MiT Stage1 feature
        #  - s2_fused : RGB + Env fusion
        #  - s3_fused : RGB + Env fusion
        #  - s4_fused : RGB + Env fusion
        return [s1, s2_fused, s3_fused, s4_fused]




print("✅ CMNeXt FRM/FFM 구조 + MiT-B5 Hub2Fuse Backbone 로드 완료")


✅ CMNeXt FRM/FFM 구조 + MiT-B5 Hub2Fuse Backbone 로드 완료


In [7]:
# ==========================================
# Encoder-Decoder wrapper that passes env_maps to backbone
# ==========================================
@MODELS.register_module()
class CMNeXtEncoderDecoder(EncoderDecoder):
    @staticmethod
    def _stack_env(data_samples):
        # (B,48,64,64)로 쌓음
        assert all(hasattr(ds, 'env_maps') for ds in data_samples), "env_maps missing in data_samples"
        return torch.stack([ds.env_maps for ds in data_samples], dim=0)

    def _forward_img_env(self, inputs, data_samples):
        env = self._stack_env(data_samples).to(inputs.device)
        # 학습/추론 공통: 백본이 (img, env) 둘 다 받음
        return self.backbone(inputs, env)

    # ---- 학습 경로(가장 중요) ----
    def loss(self, inputs, data_samples, train_cfg=None):
        feats = self._forward_img_env(inputs, data_samples)
        return self.decode_head.loss(feats, data_samples, train_cfg)

    # ---- 추론 경로 ----
    def predict(self, inputs, data_samples, test_cfg=None):
        feats = self._forward_img_env(inputs, data_samples)
        metas = [ds.metainfo for ds in data_samples]
        seg_logits = self.decode_head.predict(feats, metas, self.test_cfg if test_cfg is None else test_cfg)
        return self.postprocess_result(seg_logits, data_samples)


## 5. Training Configuration

In [8]:
# ==========================================
# MMSegmentation Config
# ==========================================

cfg = Config(dict(
    default_scope='mmseg',
    backend_args=None,
    work_dir=WORK_DIR,
    
    # Data preprocessor
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        bgr_to_rgb=False,
        pad_val=0,
        seg_pad_val=255,
        size=None,
        size_divisor=32
    ),
    
    # Model
    model=dict(
        type='CMNeXtEncoderDecoder',
        data_preprocessor=dict(
            type='SegDataPreProcessor',
            bgr_to_rgb=False,
            pad_val=0,
            seg_pad_val=255,
            size=None,
            size_divisor=32
        ),
        
        backbone=dict(
            type='CMNeXtCascadingBackbone',
            in_channels=4,              # RGBN → 내부 1×1로 RGB
            aux_c2=128, aux_c3=320, aux_c4=512,
            track_frm_stats=True,
            mit_b5_cfg=dict(       
                type='MixVisionTransformer',
                in_channels=3,
                embed_dims=64,
                num_stages=4,
                num_layers=[3, 6, 40, 3],  # B5
                num_heads=[1, 2, 5, 8],
                patch_sizes=[7, 3, 3, 3],
                sr_ratios=[8, 4, 2, 1],
                out_indices=(0, 1, 2, 3),
                mlp_ratio=4,
                qkv_bias=True,
                drop_rate=0.0,
                attn_drop_rate=0.0,
                drop_path_rate=0.1,
                init_cfg=dict(
                    type='Pretrained',
                    checkpoint='https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b5_20220624-658746d9.pth'
                )
            )
        ),
        decode_head=dict(
            type='SegformerHead',
            in_channels=[64, 128, 320, 512],
            channels=768,
            dropout_ratio=0.1,
            num_classes=2,
            threshold=0.5,
            norm_cfg=dict(type='SyncBN', requires_grad=True),
            align_corners=False,
            out_channels=1,
            loss_decode = [
            # ① BCE (binary)
            dict(
                type='CrossEntropyLoss',
                use_sigmoid=True,
                loss_name='loss_ce',
                loss_weight=1.0,
                class_weight=[2.92],
            ),

            # ② Lovasz (binary IoU surrogate)
            dict(
                type='LovaszLoss',
                loss_type='binary',
                per_image=False,
                reduction='none', 
                loss_weight=1.0,
                loss_name='loss_lovasz',
            ),
        ],
            in_index=(0, 1, 2, 3)
        ),


        train_cfg=dict(),
        test_cfg=dict(mode='whole'),
    ),
    
        
    # Data pipelines
    train_pipeline=[
        dict(type='LoadMultiModalData', extra_data_root=EXTRA_DATA_ROOT),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(
          type='NormalizeByKey',
          specs={
            'img':      (IMG_MEAN, IMG_STD),
            'env_co':   (CO_MEAN_12,   CO_STD_12),    # 길이 12
            'env_no2':  (NO2_MEAN_12,  NO2_STD_12),   # 길이 12
            'env_so2':  (SO2_MEAN_12,  SO2_STD_12),   # 길이 12
            'env_gems': (GEMS_MEAN_12, GEMS_STD_12),  # 길이 12
          }
        )
        ,
        dict(
            type='PackSegInputsKeepEnv',
            env_keys=['env_co','env_no2','env_so2','env_gems'],
            out_key='env_maps',
        ),
    ],

    val_pipeline=[
        dict(type='LoadMultiModalData', extra_data_root=EXTRA_DATA_ROOT),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(
          type='NormalizeByKey',
          specs={
            'img':      (IMG_MEAN, IMG_STD),
            'env_co':   (CO_MEAN_12,   CO_STD_12),    # 길이 12
            'env_no2':  (NO2_MEAN_12,  NO2_STD_12),   # 길이 12
            'env_so2':  (SO2_MEAN_12,  SO2_STD_12),   # 길이 12
            'env_gems': (GEMS_MEAN_12, GEMS_STD_12),  # 길이 12
          }
        )
        ,
        dict(
            type='PackSegInputsKeepEnv',
            env_keys=['env_co','env_no2','env_so2','env_gems'],
            out_key='env_maps',
        ),
    ],

    test_pipeline=[
        dict(type='LoadMultiModalData', extra_data_root=EXTRA_DATA_ROOT),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(
          type='NormalizeByKey',
          specs={
            'img':      (IMG_MEAN, IMG_STD),
            'env_co':   (CO_MEAN_12,   CO_STD_12),    # 길이 12
            'env_no2':  (NO2_MEAN_12,  NO2_STD_12),   # 길이 12
            'env_so2':  (SO2_MEAN_12,  SO2_STD_12),   # 길이 12
            'env_gems': (GEMS_MEAN_12, GEMS_STD_12),  # 길이 12
          }
        )
        ,
        dict(
            type='PackSegInputsKeepEnv',
            env_keys=['env_co','env_no2','env_so2','env_gems'],
            out_key='env_maps',
        ),
    ],
    
    # Dataloaders
    metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
    
    train_dataloader=dict(
        batch_size=16,
        num_workers=12,
        persistent_workers=True,
        sampler=dict(type='InfiniteSampler', shuffle=True),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(img_path='img_dir/train', seg_map_path='ann_dir/train'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    val_dataloader=dict(
        batch_size=16,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(img_path='img_dir/val', seg_map_path='ann_dir/val'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    test_dataloader=dict(
        batch_size=16,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(img_path='img_dir/val', seg_map_path='ann_dir/val'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    # Evaluators
    val_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    test_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    
    # Optimizer
    optim_wrapper = dict(
        type='AmpOptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=1e-4,
            betas=(0.9, 0.999),
            weight_decay=0.01,
        ),
        loss_scale='dynamic',
        paramwise_cfg=dict(
            # ✔ 모든 Norm 계열 (MiT + ENV 인코더 BN) WD=0
            norm_decay_mult=0.0,
            custom_keys={
                'pos_block': dict(decay_mult=0.0),  # MiT positional block WD=0
                'head': dict(lr_mult=10.0),         # 디코더 헤드 LR×10
            }
        )
    ),

    # Learning rate scheduler
    param_scheduler=[
        dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=3000),
        dict(type='PolyLR', begin=3000, end=200000, eta_min=1e-7, power=0.9, by_epoch=False)
    ],

    # Training loop
    train_cfg=dict(type='IterBasedTrainLoop', max_iters=MAX_ITERS, val_interval=VAL_INTERVAL),
    val_cfg=dict(type='ValLoop'),
    test_cfg=dict(type='TestLoop'),
    
    # Hooks
    default_hooks=dict(
        logger=dict(type='LoggerHook', interval=LOG_INTERVAL, log_metric_by_epoch=False),
        param_scheduler=dict(type='ParamSchedulerHook'),
        timer=dict(type='IterTimerHook'),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='SegVisualizationHook', draw=False),
        checkpoint=dict(
            type='CheckpointHook',
            by_epoch=False,
            interval=CHECKPOINT_INTERVAL,
            max_keep_ckpts=3,
            save_best='mIoU'
        )
    ),
    
    # Visualization backends (with Wandb)
    vis_backends=[
        dict(type='LocalVisBackend', save_dir=WORK_DIR),
#         dict(
#             type='WandbVisBackend',
#             init_kwargs=dict(
#                 project=WANDB_PROJECT,
#                 name=WANDB_NAME,
#                 entity=WANDB_ENTITY,
#                 config=dict(
#                     model='SegFormer-B5',
#                     decoder='CMNeXtFusion',
#                     adapter='1x1Conv-RGBIdentity',
#                     input_channels=4,
#                     env_channels=48,
#                     max_iters=MAX_ITERS,
#                     batch_size=8,
#                     learning_rate=6e-5,
#                     seed=SEED,
#                 )
#             )
#         ),
    ],
    
    visualizer=dict(
        type='SegLocalVisualizer',
        vis_backends=[
            dict(type='LocalVisBackend', save_dir=WORK_DIR),
#             dict(
#                 type='WandbVisBackend',
#                 init_kwargs=dict(
#                     project=WANDB_PROJECT,
#                     name=WANDB_NAME,
#                     entity=WANDB_ENTITY,
#                 )
#             ),
        ],
        name='visualizer'
    ),
    
    # Environment
    env_cfg=dict(
        cudnn_benchmark=True,
        mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
        dist_cfg=dict(backend='nccl')
    ),
    
    # Randomness
    randomness=dict(seed=SEED, deterministic=False, diff_rank_seed=False),
    
    launcher='none',
    log_level='INFO',
))

# Connect pipelines
cfg.train_dataloader.dataset.pipeline = cfg.train_pipeline
cfg.val_dataloader.dataset.pipeline = cfg.val_pipeline
cfg.test_dataloader.dataset.pipeline = cfg.test_pipeline
cfg.load_from = '/home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/merge_fixed.pth'



print("\n" + "="*60)
print("✅ Configuration Complete")
print("="*60)
print(f"Model: SegFormer MiT-B5 + CMNeXt Fusion")
print(f"Input: 4-channel (RGBN) + 48-channel (Env)")
print(f"Max Iterations: {MAX_ITERS:,}")
print(f"Validation Interval: {VAL_INTERVAL:,}")
# print(f"Wandb Logging: Enabled")
print(f"Random Seed: {SEED}")
print("="*60)


✅ Configuration Complete
Model: SegFormer MiT-B5 + CMNeXt Fusion
Input: 4-channel (RGBN) + 48-channel (Env)
Max Iterations: 200,000
Validation Interval: 1,000
Random Seed: 42


## 6. Training

In [12]:
# ==========================================
# Start Training
# ==========================================

print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)

runner = Runner.from_cfg(cfg)
runner.train()

print("\n" + "="*60)
print("✅ TRAINING COMPLETE")
print("="*60)
print(f"Results saved to: {WORK_DIR}")
# print(f"Wandb run: {WANDB_PROJECT}/{WANDB_NAME}")


🚀 STARTING TRAINING
11/24 06:17:27 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_52,code=sm_52;-ge

Downloading: "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b5_20220624-658746d9.pth" to /home/work/.cache/torch/hub/checkpoints/mit_b5_20220624-658746d9.pth


Loads checkpoint by local backend from path: /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/merge_fixed.pth
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.stems.0.s2.0.weight, backbone.stems.0.s2.1.weight, backbone.stems.0.s2.1.bias, backbone.stems.0.s2.1.running_mean, backbone.stems.0.s2.1.running_var, backbone.stems.0.s2.3.weight, backbone.stems.0.s2.4.weight, backbone.stems.0.s2.4.bias, backbone.stems.0.s2.4.running_mean, backbone.stems.0.s2.4.running_var, backbone.stems.0.s3.0.weight, backbone.stems.0.s3.1.weight, backbone.stems.0.s3.1.bias, backbone.stems.0.s3.1.running_mean, backbone.stems.0.s3.1.running_var, backbone.stems.0.s4.0.weight, backbone.stems.0.s4.1.weight, backbone.stems.0.s4.1.bias, backbone.stems.0.s4.1.running_mean, backbone.stems.0.s4.1.running_var, backbone.stems.1.s2.0.weight, backbone.stems.1.s2.1.weight, backbone.stems.1.s2.1.bias, backbone.stems.1.s2.1.running_mean, backbone.stems.1.s2.1.running_var

## 7. Evaluation (Optional)

In [10]:
import os
from mmengine.config import Config
from mmengine.runner import Runner

WORK_DIR  = "/home/work/root/Meme_coin/ddc/work_dirs/segformer_b5_cmnext_m4_final"

CFG_PATH  = os.path.join(WORK_DIR, "config.py")                 # config.py 사용
CKPT_PATH = os.path.join(WORK_DIR, "best_mIoU_iter_199000.pth")  # .pth 체크포인트

print("CFG_PATH :", CFG_PATH)
print("CKPT_PATH:", CKPT_PATH)

cfg = Config.fromfile(CFG_PATH)
cfg.load_from = CKPT_PATH       # 여기서 best ckpt 로드
cfg.work_dir = WORK_DIR
cfg.launcher = 'none'           # 단일 GPU면 이렇게

runner = Runner.from_cfg(cfg)

# 모델 핸들
model = runner.model

# ---- 평가도 같이 돌리고 싶으면 ----
test_metrics = runner.test()
print("Test metrics:", test_metrics)


CFG_PATH : /home/work/root/Meme_coin/ddc/work_dirs/segformer_b5_cmnext_m4_final/config.py
CKPT_PATH: /home/work/root/Meme_coin/ddc/work_dirs/segformer_b5_cmnext_m4_final/best_mIoU_iter_199000.pth
11/26 02:04:50 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is en

In [11]:
# ---- SelfQueryHub 통계 확인  ----
hub2 = model.backbone.sq2
if hub2.total_positions > 0:
    freq2 = (hub2.select_counts / hub2.total_positions).cpu().numpy()
    print("S2 modal selection freq:", freq2)
else:
    print("SelfQueryHub S2 stats not collected yet.")
    
hub3 = model.backbone.sq3
if hub3.total_positions > 0:
    freq3 = (hub3.select_counts / hub3.total_positions).cpu().numpy()
    print("S3 modal selection freq:", freq3)
else:
    print("SelfQueryHub S3 stats not collected yet.")
    
hub4 = model.backbone.sq4
if hub4.total_positions > 0:
    freq4 = (hub4.select_counts / hub4.total_positions).cpu().numpy()
    print("S4 modal selection freq:", freq4)
else:
    print("SelfQueryHub S4 stats not collected yet.")

S2 modal selection freq: [0.25493434 0.03966382 0.11185791 0.59354395]
S3 modal selection freq: [0.64767283 0.06959765 0.10702637 0.17570312]
S4 modal selection freq: [0.01197656 0.20702344 0.26820704 0.51279294]


In [12]:
# ---- FRM 통계 확인 (S2/S3/S4) ----

frm2 = model.backbone.frm2
if frm2.frm_calls > 0:
    env2rgb_2 = (frm2.env_to_rgb_mag / frm2.frm_calls).item()
    rgb2env_2 = (frm2.rgb_to_env_mag / frm2.frm_calls).item()
    print(f"S2 Env→RGB mag(avg): {env2rgb_2:.6f}")
    print(f"S2 RGB→Env mag(avg): {rgb2env_2:.6f}")
else:
    print("FRM2 stats not collected yet. (frm_calls=0)")


frm3 = model.backbone.frm3
if frm3.frm_calls > 0:
    env2rgb_3 = (frm3.env_to_rgb_mag / frm3.frm_calls).item()
    rgb2env_3 = (frm3.rgb_to_env_mag / frm3.frm_calls).item()
    print(f"S3 Env→RGB mag(avg): {env2rgb_3:.6f}")
    print(f"S3 RGB→Env mag(avg): {rgb2env_3:.6f}")
else:
    print("FRM3 stats not collected yet. (frm_calls=0)")


frm4 = model.backbone.frm4
if frm4.frm_calls > 0:
    env2rgb_4 = (frm4.env_to_rgb_mag / frm4.frm_calls).item()
    rgb2env_4 = (frm4.rgb_to_env_mag / frm4.frm_calls).item()
    print(f"S4 Env→RGB mag(avg): {env2rgb_4:.6f}")
    print(f"S4 RGB→Env mag(avg): {rgb2env_4:.6f}")
else:
    print("FRM4 stats not collected yet. (frm_calls=0)")


S2 Env→RGB mag(avg): 0.009627
S2 RGB→Env mag(avg): 0.449017
S3 Env→RGB mag(avg): 0.002548
S3 RGB→Env mag(avg): 0.417124
S4 Env→RGB mag(avg): 0.009355
S4 RGB→Env mag(avg): 0.815433
